<a href="https://colab.research.google.com/github/BdeJMM/Proyecto_Gestion_datos_IA/blob/main/Creacion_de_datos_EV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import logging

# Configurar logs (evidencia para el informe)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s'
)
log = logging.getLogger("pipeline_fraude")

log.info("Iniciando pipeline de detección de fraude")

np.random.seed(42)
random.seed(42)

N_LEGIT  = 9000
N_FRAUD  = 500

categorias   = ['food_dining','shopping_net','entertainment',
                'gas_transport','grocery_pos','misc_net','travel']
jobs         = ['Engineer','Teacher','Doctor','Accountant',
                'Manager','Nurse','Analyst']
merch_legit  = ['Walmart','Amazon','Starbucks','Shell','Netflix']
merch_fraud  = ['Unknown_XZ','Offshore99','FastCash_Net','ShadyDeals']

def fecha_aleatoria(inicio, fin):
    delta = fin - inicio
    return inicio + timedelta(seconds=random.randint(0, int(delta.total_seconds())))

filas = []
inicio = datetime(2023, 1, 1)
fin    = datetime(2023, 12, 31)

# --- Transacciones legítimas ---
for i in range(N_LEGIT):
    dt  = fecha_aleatoria(inicio, fin)
    lat = round(random.uniform(25, 48), 4)
    lon = round(random.uniform(-120, -70), 4)
    cat = random.choice(categorias)
    amt = round(random.uniform(5, 300), 2)

    filas.append({
        'trans_date_trans_time': dt.strftime('%Y-%m-%d %H:%M:%S'),
        'unix_time' : int(dt.timestamp()),
        'cc_num'    : random.randint(10**15, 10**16 - 1),
        'merchant'  : random.choice(merch_legit),
        'category'  : cat,
        'amt'       : amt,
        'gender'    : random.choice(['M','F']),
        'city_pop'  : random.randint(50_000, 3_000_000),
        'job'       : random.choice(jobs),
        'lat'       : lat,
        'long'      : lon,
        # Comercio cerca del titular → legítimo
        'merch_lat' : round(lat + random.uniform(-0.5, 0.5), 4),
        'merch_long': round(lon + random.uniform(-0.5, 0.5), 4),
        'is_fraud'  : 0
    })

# --- Transacciones fraudulentas ---
for i in range(N_FRAUD):
    dt  = fecha_aleatoria(inicio, fin)
    dt  = dt.replace(hour=random.choice([0,1,2,3,23]))  # hora nocturna
    lat = round(random.uniform(25, 48), 4)
    lon = round(random.uniform(-120, -70), 4)

    filas.append({
        'trans_date_trans_time': dt.strftime('%Y-%m-%d %H:%M:%S'),
        'unix_time' : int(dt.timestamp()),
        'cc_num'    : random.randint(10**15, 10**16 - 1),
        'merchant'  : random.choice(merch_fraud),
        'category'  : random.choice(['shopping_net','misc_net','travel']),
        'amt'       : round(random.uniform(800, 5000), 2),  # monto alto
        'gender'    : random.choice(['M','F']),
        'city_pop'  : random.randint(50_000, 3_000_000),
        'job'       : random.choice(jobs),
        'lat'       : lat,
        'long'      : lon,
        # Comercio MUY lejos → fraude
        'merch_lat' : round(random.uniform(25, 48), 4),
        'merch_long': round(random.uniform(-120, -70), 4),
        'is_fraud'  : 1
    })

df_raw = pd.DataFrame(filas).sample(frac=1).reset_index(drop=True)

log.info(f"Dataset generado: {len(df_raw)} filas")
log.info(f"Distribución: {df_raw['is_fraud'].value_counts().to_dict()}")
log.info(f"Columnas: {list(df_raw.columns)}")

df_raw.head()